# Lab 4: Parameter-efficient fine-tuning

Fine-tuning all parameters of pre-trained language models can be resource-intensive. Because of this, current research in natural language processing is looking into developing methods for adapting models to downstream tasks without full fine-tuning. These methods only tune a small number of model parameters while yielding performance comparable to that of a fully fine-tuned model.

In this lab, you will implement LoRA, one of the most well-known methods for parameter-efficient fine-tuning. LoRA stands for “Low-Rank Adaptation of Large Language Models” and was originally described in a research article by [Hu et al. (2021)](https://arxiv.org/abs/2106.09685).

Along the way, you will earn experience with [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index), a state-of-the-art library for training and deploying language models, as well as with several related libraries. In particular, you will learn a best-practice workflow for downloading a Transformer model and fine-tuning it on the downstream task of binary sentiment classification.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Dataset

The data for this lab comes from the [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/). The full dataset consists of 50,000 highly polar movie reviews collected from the Internet Movie Database (IMDB). Here, we use a random sample consisting of 2,000 reviews for training and 500 reviews for evaluation.

To load the dataset, we use the [Hugging Face Datasets](https://huggingface.co/docs/datasets/en/index) library.

In [1]:
from datasets import load_dataset

imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 500
    })
})

As we can see, each sample in the dataset is a record with three fields: an internal index (`index`, an integer), the text of the review (`review`, a string), and the sentiment label (`label`, an integer – 1&nbsp;for “positive” and 0&nbsp;for “negative” sentiment).

Here is an example record:

In [2]:
imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

## Tokeniser

As our pre-trained language model, we will use [DistilBERT](https://huggingface.co/docs/transformers/en/model_doc/distilbert), a compact encoder model with 40% less parameters than BERT base. DistilBERT is not actually a *large* language model by modern standards and thus does not benefit as much from parameter-efficient fine-tuning as other models. However, it has the benefit of being light and fast, and can be run even on consumer hardware.

To feed the movie reviews to DistilBERT, we need to tokenise them and encode the resulting tokens as integers in the model vocabulary. We start by loading the DistilBERT tokeniser using the [Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto):

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

We then create a tokenised version of the dataset:

In [4]:
def tokenize_function(batch):
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'attention_mask'],
        num_rows: 500
    })
})

As we can see, tokenising adds two additional fields to each review: `input_ids` is the list of token ids corresponding to the review, and `attention_mask` is the list of indices specifying which tokens the encoder should attend to.

To avoid trouble when fine-tuning the model later, the next cell disables tokeniser parallelism.

In [5]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Trainer

In this section, we will set up our workflow for training and evaluating DistilBERT models. The central component in this workflow is the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), which provides extensive configuration options. Here, we leave most of these options at their default value. Two changes we *do* make are to enable evaluation of the trained model after each epoch, and to log the training and evaluation loss after every 5&nbsp;training steps (the default is 500).

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    logging_steps=5,
)

In addition to the loss, we also track classification accuracy. For this we import the [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/en/index) library and define a small helper function `compute_metrics()` that the trainer will call after each epoch.

In [7]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In the next cell we define a convenience function `make_trainer()` that creates a readily-configured trainer for a specified model (*model*). We will use this trainer both to train the model on the training section of the tokenised review dataset, and to evaluate it on the evaluation section.

In [8]:
from transformers import Trainer


def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_imdb_dataset["train"],
        eval_dataset=tokenized_imdb_dataset["eval"],
        compute_metrics=compute_metrics,
    )
    return trainer

## Full fine-tuning

In the rest of this notebook, we will work our way to the implementation of LoRA, and compare LoRA to traditional fine-tuning methods. Our first point of reference is a fully fine-tuned DistilBERT model.

We start by loading the pre-trained model:

In [9]:
from transformers import AutoModelForSequenceClassification

pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

pretrained_model

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(p

The architecture of DistilBERT is that of a standard Transformer encoder with an embedding layer (`embeddings`) followed by a stack of six Transformer blocks (`transformer`) and a feedforward network with two linear layers (`pre_classifier` and `classifier`) and a final dropout layer (`dropout`).

### 🧩 Task 4.01: Counting the number of trainable parameters

One relevant measure in the context of parameter-efficient fine-tuning is the number of parameters that need to be changed when training a model. Your first task in this lab is to write a function `num_trainable_parameters()` that calculates this number for a given model.

In [10]:
def num_trainable_parameters(model):
    # Count all scalar parameters that will be updated during training.
    # Sum the number of elements in all parameters that require gradients.
    # p.numel() returns the total number of scalar values in a parameter tensor.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

The function should implement the following specification:

> **num_trainable_parameters** (*model*)
>
> Returns the number of float-valued trainable parameters in the specified *model* as an integer.

#### 👍 Hint

The term *parameter* can refer to either complete tensors or the individual elements of these tensors. For example, a linear layer created by `nn.Linear(3, 5)` has 2&nbsp;tensor-valued parameters (a weight matrix and a bias vector) and 20&nbsp;float-valued parameters (the elements of these tensors). To get the tensor-valued parameters of a model, you can use the [`parameters()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters) method. A parameter is *trainable* if it requires gradient.

#### 🤞 Test your code

To test your code, apply your function to the pre-trained model. The correct number of float-valued trainable parameters for this model is 66,955,010.

In [11]:
num_trainable_parameters(pretrained_model)

66955010

### Fine-tuning

When we load the pre-trained model, the Hugging Face Transformers library warns us that the weights of the feedforward network have not yet been trained. To do so, in the next cell, we pass the pre-trained model to a trainer and initiate the fine-tuning process.

**⚠️ Please note that fine-tuning the model will take some time! ⚠️**

You can work on the other problems in this lab while you are waiting.

In [12]:
finetuned_trainer = make_trainer(pretrained_model)

finetuned_trainer.train()

  0%|          | 0/750 [00:00<?, ?it/s]/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  1%|          | 6/750 [00:01<03:45,  3.29it/s]

{'loss': 0.6737, 'grad_norm': 1.7256743907928467, 'learning_rate': 4.966666666666667e-05, 'epoch': 0.02}


  1%|▏         | 11/750 [00:03<03:43,  3.31it/s]

{'loss': 0.7092, 'grad_norm': 1.2210239171981812, 'learning_rate': 4.933333333333334e-05, 'epoch': 0.04}


  2%|▏         | 16/750 [00:04<03:39,  3.35it/s]

...

100%|██████████| 750/750 [03:59<00:00,  3.33it/s]

{'loss': 0.0777, 'grad_norm': 0.06039462611079216, 'learning_rate': 0.0, 'epoch': 3.0}


                                                 
100%|██████████| 750/750 [04:03<00:00,  3.08it/s]

{'eval_loss': 0.44649651646614075, 'eval_accuracy': 0.902, 'eval_runtime': 3.9456, 'eval_samples_per_second': 126.723, 'eval_steps_per_second': 15.967, 'epoch': 3.0}
{'train_runtime': 243.2076, 'train_samples_per_second': 24.67, 'train_steps_per_second': 3.084, 'train_loss': 0.22222236871098477, 'epoch': 3.0}


TrainOutput(global_step=750, training_loss=0.22222236871098477, metrics={'train_runtime': 243.2076, 'train_samples_per_second': 24.67, 'train_steps_per_second': 3.084, 'total_flos': 794804391936000.0, 'train_loss': 0.22222236871098477, 'epoch': 3.0})

Because full fine-tuning is so resource-intensive, we save the fine-tuned model to disk:

In [13]:
finetuned_trainer.save_model("finetuned")

Later in this notebook, whenever you need the fully fine-tuned version of the model, you can load it as follows:

In [14]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

### Convenience functions

Because we will repeat the steps we just took to fine-tune the pre-trained model several times in this notebook, we define two convenience functions:

In [15]:
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model

In [16]:
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

## Tuning the final layers only

If full fine-tuning marks one end of the complexity spectrum, the other end is marked by only tuning the final layers of the transformer – the *head* of the model. In the case of DistilBERT, the head consists of the `pre_classifier` and `classifier` layers.

### 🧩 Task 4.02: Head-tuning

Implement the head-tuning strategy by coding the following function:

In [17]:
def make_headtuned_model():
    # Load a fresh copy of the pre-trained DistilBERT model.
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    # Freeze all model parameters.
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the classification head.
    for param in model.pre_classifier.parameters():
        param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    # Train only the unfrozen parameters.
    return train(model)

Here is the specification of this function:

> **make_headtuned_model** ()
>
> Returns a model that is identical to the pre-trained model, except that the head layers have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.)

#### 👍 Hint

You freeze a parameter by setting its `requires_grad`-attribute to `False`.

Once you have an implementation of the head-tuning strategy, evaluate it on the evaluation data. How much accuracy do we lose when only training the final layers of the pre-trained model, compared to full fine-tuning?

In [18]:
headtuned_model = make_headtuned_model()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of trainable parameters: 592130


  0%|          | 0/750 [00:00<?, ?it/s]/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  1%|          | 6/750 [00:00<00:53, 14.02it/s]

{'loss': 0.6907, 'grad_norm': 2.368324041366577, 'learning_rate': 4.966666666666667e-05, 'epoch': 0.02}


  2%|▏         | 12/750 [00:00<00:54, 13.65it/s]

...

{'loss': 0.4924, 'grad_norm': 2.2950518131256104, 'learning_rate': 3.3333333333333335e-07, 'epoch': 2.98}


100%|██████████| 750/750 [01:06<00:00, 12.81it/s]

{'loss': 0.5158, 'grad_norm': 2.7998297214508057, 'learning_rate': 0.0, 'epoch': 3.0}



100%|██████████| 750/750 [01:10<00:00, 10.69it/s]

{'eval_loss': 0.5114938020706177, 'eval_accuracy': 0.81, 'eval_runtime': 3.9624, 'eval_samples_per_second': 126.187, 'eval_steps_per_second': 15.9, 'epoch': 3.0}
{'train_runtime': 70.154, 'train_samples_per_second': 85.526, 'train_steps_per_second': 10.691, 'train_loss': 0.5764305334091187, 'epoch': 3.0}


#### 🤞 Test your code

If you configured your model correctly, `num_trainable_parameters()` should show 592,130 trainable parameters.

For future reference, we also save the head-tuned model:

In [19]:
make_trainer(headtuned_model).save_model("headtuned")

#### REMARK - Task 4.02

The fully fine-tuned model achieved an evaluation accuracy of **90.2%**, while
the head-tuned model achieved **81.0%**. Thus, training only the classification
head results in a decrease of **9.2 percentage points** in accuracy.

However, head tuning requires only **592,130 trainable parameters**, compared
with **66,955,010** for full fine-tuning. This corresponds to only about
**0.88%** of the parameters used in full fine-tuning, or a reduction of roughly
**99.1%**.

## Layer surgery

LoRA works by “wrapping” frozen layers from the pre-trained Transformer model inside adapter modules. Conventionally, this wrapping is only applied to the linear layers that transform the queries and values in the self-attention mechanism. To implement the wrapping, we need functions to extract and replace layers in a model. Your task in this section is to code these functions.

### 🎓 Task 4.03: Extracting layers

Code a function that extracts the query and value linear layers from a DistilBERT model:

In [23]:
def extract(model):
    named_layers = {}

    # Extract the query and value projections from every Transformer block.
    for i in range(6):
        q_name = f"distilbert.transformer.layer.{i}.attention.q_lin"
        v_name = f"distilbert.transformer.layer.{i}.attention.v_lin"

        named_layers[q_name] = model.get_submodule(q_name)
        named_layers[v_name] = model.get_submodule(v_name)

    return named_layers

Implement this function to match the following specification:

> **extract** (*model*)
>
> Takes a DistilBERT model (*model*) and extracts the query and value linear layers from each block of the Transformer. Returns a dictionary mapping the DistilBERT module names of these layers to the layers themselves (instances of `nn.Linear`).

#### 👍 Hint

As we saw earlier, the DistilBERT model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. Use [`get_submodule()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.get_submodule) to retrieve a layer by name. You can hard-wire the names of the layers you want to extract.

#### 🤞 Test your code

To test your code, check the number of trainable float-valued parameters in the extracted layers. This number should be 7,087,104.

In [24]:
extracted_layers = extract(pretrained_model)

extracted_layers

{'distilbert.transformer.layer.0.attention.q_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.0.attention.v_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.1.attention.q_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.1.attention.v_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.2.attention.q_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.2.attention.v_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.3.attention.q_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.3.attention.v_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.4.attention.q_lin': Linear(in_features=768, out_features=768, bias=True),
 'distilbert.transformer.layer.4.attention.v_lin': Line

In [25]:
sum(
    p.numel()
    for layer in extracted_layers.values()
    for p in layer.parameters()
    if p.requires_grad
)

7087104

### 🎓 Task 4.04: Replacing layers

Next, code the inverse of the `extract()` function to replace selected layers of a module using a dictionary of named layers.

In [26]:
def replace(model, named_layers):
    for name, layer in named_layers.items():
        # Split the fully-qualified module name into:
        # - the parent module path
        # - the final attribute name
        parent_name, child_name = name.rsplit(".", 1)

        # Retrieve the parent module.
        parent = model.get_submodule(parent_name)

        # Replace the selected child layer.
        setattr(parent, child_name, layer)

    return model

Implement this function to match the following specification:

> **replace** (*model*, *named_layers*)
>
> Takes a DistilBERT model (*model*) and a dictionary in the format returned by `extract()` (*named_layers*) and injects the extracted layers into the model. More specifically, suppose that *named_layers* contains a key–value pair `(name, layer)`. Then the function replaces the submodule of *model* addressed by the fully-qualified string name `name` by the layer `layer`. Returns the modified model.

#### 👍 Hint

Use [`getattr()`](https://docs.python.org/3/library/functions.html#getattr) and [`setattr()`](https://docs.python.org/3/library/functions.html#setattr) to return or set the value of a named submodule.

#### 🤞 Test your code

To test your implementation, write code that (1)&nbsp;extracts the query and value linear layers from the fine-tuned model; (2)&nbsp;replaces these layers with clones with random weights; and (3)&nbsp;replaces these layers again with the original versions. Evaluating the modified model after step&nbsp;(2) should yield a near-random accuracy. Evaluating it again after step&nbsp;(3) should yield the original accuracy.

The following function should be helpful. It clones a linear layer, copying the weights and the bias from the original.

In [27]:
import torch.nn as nn


def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy

In [28]:
# Extract the original query and value layers from the fine-tuned model.
original_layers = extract(finetuned_model)

# Create fresh randomly initialised linear layers with the same dimensions.
random_layers = {}

for name, layer in original_layers.items():
    out_features, in_features = layer.weight.shape
    random_layers[name] = nn.Linear(in_features, out_features)

# Replace the fine-tuned layers with random ones.
replace(finetuned_model, random_layers)

# Accuracy should now be much worse, typically close to random classification.
evaluate(finetuned_model)

/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.57it/s]


{'eval_loss': 0.7107182741165161,
 'eval_accuracy': 0.474,
 'eval_runtime': 3.4757,
 'eval_samples_per_second': 143.858,
 'eval_steps_per_second': 18.126}

In [29]:
replace(finetuned_model, original_layers)

# Accuracy should return to approximately the original fine-tuned accuracy.
evaluate(finetuned_model)

/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.64it/s]


{'eval_loss': 0.44649651646614075,
 'eval_accuracy': 0.902,
 'eval_runtime': 3.4741,
 'eval_samples_per_second': 143.922,
 'eval_steps_per_second': 18.134}

#### REMARK - Task 4.04

Replacing the fine-tuned query and value layers with randomly initialised layers reduced the evaluation accuracy from **90.2%** to **47.4%**, which is close to random performance for binary classification.

After restoring the original layers, the accuracy returned to **90.2%**. This confirms that the `extract()` and `replace()` functions correctly identify and modify the intended DistilBERT attention layers.

## Low-rank approximation

The basic idea behind LoRA is to conceptualise fine-tuned weights as a sum $W_0 + \Delta W$ of the weights from the pre-trained model, $W_0$, and a low-rank update matrix $\Delta W$. The goal of fine-tuning, then, is to learn the update matrix; this happens in the adapter layers.

Before we get to the implementation of the LoRA adapter layers, we first check to what extent the assumption that fine-tuning can be described by low-rank matrices holds true for DistilBERT. To do so, we will “cheat” and replace the query and value linear layers of the head-tuned model with low-rank approximations. The technical key to this is the truncated singular value decomposition (SVD).

### 🎓 Task 4.05: Low-rank matrix approximation

Your first task in this section is to implement the low-rank matrix approximation.

In [ ]:
import torch

def approximate(matrix, rank):
    # Compute the singular value decomposition:
    # matrix = U @ diag(S) @ Vh
    U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)

    # Keep only the first `rank` singular components.
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]

    # Reconstruct the rank-r approximation.
    return U_r @ torch.diag(S_r) @ Vh_r

Implement this function to match the following specification:

> **approximate** (*matrix*, *rank*)
>
> Takes a 2D-tensor (*matrix*) and an integer rank $r$ (*rank*), computes the truncated SVD with rank $r$ on the tensor, and returns the corresponding low-rank approximation matrix.

#### 👍 Hint

If you need a refresher on the low-rank matrix approximation, read the corresponding section from the Wikipedia article on the [Singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition#Low-rank_matrix_approximation). The truncated SVD is an extension of the full SVD; the latter can be computed using [`torch.linalg.svd()`](https://pytorch.org/docs/stable/generated/torch.linalg.svd.html).

#### 🤞 Test your code

To test your code, run the following cell. It creates a matrix `original` with rank $r \leq 8$ and after that the rank-$8$ approximation matrix `approximation`. You should find that the distance between the two matrices is very low.

In [34]:
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
torch.dist(original, approximation)

tensor(0.0005)

#### REMARK - Task 4.05

The distance between the original matrix and its rank-8 approximation is approximately **0.0005**, so the approximation is almost identical to the original matrix.

The original matrix has shape $768 \times 384$, but it was constructed through an intermediate dimension of 8, so its rank is at most 8. This means that although the matrix is large, it can be represented using only 8 independent components.

### 🎓 Task 4.06: Approximated fine-tuned model (version 1)

In the next step, your task is to construct a version of the head-tuned model in which every query and value linear layer is replaced by a low-rank approximation of the corresponding layer from the fully fine-tuned model.

In [35]:
def make_approximated_model_1(rank):
    # Load a fresh copy of the head-tuned model.
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")

    # Extract the query and value layers from the fully fine-tuned model.
    finetuned_layers = extract(finetuned_model)

    approximated_layers = {}

    for name, layer in finetuned_layers.items():
        # Clone the fine-tuned layer so that we do not modify the original.
        approximated_layer = clone_linear(layer)

        # Replace its weight matrix with a rank-r approximation.
        with torch.no_grad():
            approximated_layer.weight.copy_(
                approximate(layer.weight, rank)
            )

        approximated_layers[name] = approximated_layer

    # Insert the approximated layers into the head-tuned model.
    return replace(model, approximated_layers)

Here is the specification of this function:

> **make_approximated_model_1** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which every query and value linear layer is replaced by its $r$-approximated corresponding layer from the fully fine-tuned model.

Run the next cell to evaluate your model for different rank values. Start with the full rank and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [36]:
approximated_model_1 = make_approximated_model_1(768)

evaluate(approximated_model_1)

/var/folders/rn/q7z463h93xx6qvf37bry6xt80000gp/T/ipykernel_91858/2213654790.py:6: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:34.)
  U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.64it/s]


{'eval_loss': 0.3504483699798584,
 'eval_accuracy': 0.878,
 'eval_runtime': 4.2735,
 'eval_samples_per_second': 117.001,
 'eval_steps_per_second': 14.742}

In [37]:
for rank in [768, 384, 192, 96, 48, 24, 12, 6, 3]:
    model = make_approximated_model_1(rank)
    results = evaluate(model)

    print(f"Rank {rank}: accuracy = {results['eval_accuracy']:.3f}")

/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.38it/s]


Rank 768: accuracy = 0.878


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.34it/s]


Rank 384: accuracy = 0.852


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.64it/s]


...

Rank 12: accuracy = 0.588


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 17.81it/s]


Rank 6: accuracy = 0.614


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 17.54it/s]

Rank 3: accuracy = 0.530


#### REMARK - Task 4.06

The head-tuned model achieved an accuracy of **81.0%**. When replacing its query and value layers with low-rank approximations of the corresponding fully fine-tuned layers, the accuracy decreased as the rank was reduced.

The measured accuracies were:

- rank 768: **87.8%**
- rank 384: **85.2%**
- rank 192: **79.0%**
- rank 96: **78.0%**
- rank 48: **72.6%**
- rank 24: **72.0%**
- rank 12: **58.8%**
- rank 6: **61.4%**
- rank 3: **53.0%**

The lowest tested rank that still performs better than the head-tuned model is therefore **384**.

The general trend shows that reducing the rank removes increasingly important information from the fine-tuned weight matrices, which leads to lower classification accuracy. The small increase from rank 12 to rank 6 is just a minor irregularity in the evaluation results rather than the overall trend.

Even with rank 768, the model does not reach the fully fine-tuned accuracy of **90.2%** because only the query and value layers are taken from the fully fine-tuned model. The remaining Transformer layers still come from the head-tuned model.

### 🎓 Task 4.07: Approximated fine-tuned model (version 2)

In the approximated model from the previous section, the truncated SVD is applied to the full weight matrix of the fine-tuned model: $W_0 + \Delta W$. In LoRA, the low-rank approximation only applies to the *update matrix* $\Delta W$, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

In [40]:
def make_approximated_model_2(rank):
    # Load a fresh copy of the head-tuned model.
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")

    # Load a fresh copy of the original pre-trained model.
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    # Extract query and value layers.
    pretrained_layers = extract(base_model)
    finetuned_layers = extract(finetuned_model)

    approximated_layers = {}

    for name in finetuned_layers:
        pretrained_layer = pretrained_layers[name]
        finetuned_layer = finetuned_layers[name]

        # Clone the pre-trained layer.
        approximated_layer = clone_linear(pretrained_layer)

        with torch.no_grad():
            # Move W0 to the same device as the fine-tuned weights.
            w0 = pretrained_layer.weight.to(finetuned_layer.weight.device)

            # Compute the update introduced by full fine-tuning.
            delta_w = finetuned_layer.weight - w0

            # Approximate only the update matrix.
            delta_w_approx = approximate(delta_w, rank)

            # Reconstruct W0 + low-rank ΔW.
            new_weight = w0 + delta_w_approx

            # Copy the result back to the device of the cloned layer.
            approximated_layer.weight.copy_(
                new_weight.to(approximated_layer.weight.device)
            )

        approximated_layers[name] = approximated_layer

    return replace(model, approximated_layers)

Implement the function to match the following specification:

> **make_approximated_model_2** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which the weight matrix of every query and value linear layer is replaced by the sum $W_0 + \Delta W$, where $W_0$ is the weight matrix of the pre-trained model and $\Delta W$ is the rank-$r$ approximation of the update matrix, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

Run the next cell to evaluate your model for different rank values. Start with the rank from the approximated model from the previous section and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [41]:
approximated_model_2 = make_approximated_model_2(768)

evaluate(approximated_model_2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.64it/s]


{'eval_loss': 0.3517600893974304,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.4481,
 'eval_samples_per_second': 145.009,
 'eval_steps_per_second': 18.271}

In [42]:
for rank in [384, 192, 96, 48, 24, 12, 6, 3]:
    model = make_approximated_model_2(rank)
    results = evaluate(model)

    print(f"Rank {rank}: accuracy = {results['eval_accuracy']:.3f}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.47it/s]


Rank 384: accuracy = 0.878


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.06it/s]


Rank 192: accuracy = 0.876


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 18.07it/s]


...

Rank 12: accuracy = 0.872


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 17.93it/s]


Rank 6: accuracy = 0.868


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 63/63 [00:03<00:00, 17.57it/s]

Rank 3: accuracy = 0.870


#### REMARK - Task 4.07

The head-tuned model achieved an accuracy of **81.0%**. When approximating only the fine-tuning update matrix $\Delta W$, the performance remained very stable even for very small ranks.

The measured accuracies were:

- rank 768: **87.8%**
- rank 384: **87.8%**
- rank 192: **87.6%**
- rank 96: **87.6%**
- rank 48: **87.6%**
- rank 24: **87.4%**
- rank 12: **87.2%**
- rank 6: **86.8%**
- rank 3: **87.0%**

Therefore, the lowest tested rank that still performs better than the head-tuned model is **3**.

This is very different from Task 4.06, where approximating the complete fine-tuned weight matrix required a much higher rank. Here, the update matrix $\Delta W$ can be represented well with only a few components. This supports the main idea behind LoRA: the changes introduced during fine-tuning can often be captured by a low-rank update rather than modifying the full weight matrix.

## Low-Rank Adaptation (LoRA)

In this section, you will implement the LoRA adapters and fine-tune the adapted model.

### 🎓 Task 4.08: Implement the adapter

A LoRA adapter implements the forward function

$$
y = x W_0 + x \Delta W = x W_0 + x A B
$$

where $W_0$ is a linear transformation from the pre-trained model and $\Delta W$ is a learned update matrix, deconstructed into the product $AB$ of two rank-$r$ matrices $A$ and $B$. LoRA scales the update matrix $\Delta W$ by a factor of $\alpha / r$, where $\alpha$ is a hyperparameter. (To keep the formula tidy, we ignore the fact that the linear transformation in the pre-trained model may additionally include a bias.)

In [43]:
import torch.nn as nn


class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()

        # Store the original pretrained linear layer.
        self.pretrained = pretrained

        # Freeze the original pretrained weights.
        for param in self.pretrained.parameters():
            param.requires_grad = False

        # Get the input and output dimensions of the linear layer.
        in_features = pretrained.in_features
        out_features = pretrained.out_features

        # Low-rank matrices:
        # A maps from input dimension -> rank
        # B maps from rank -> output dimension
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)

        # A is initialised from a standard normal distribution.
        nn.init.normal_(self.A.weight, mean=0.0, std=1.0)

        # B is initialised with zeros.
        nn.init.zeros_(self.B.weight)

        # Scaling factor used by LoRA.
        self.scaling = alpha / rank

    def forward(self, x):
        # Original frozen transformation + learned low-rank update.
        return self.pretrained(x) + self.scaling * self.B(self.A(x))

Your code must comply with the following specification:

**__init__** (*self*, *pretrained*, *rank* = 12, *alpha* = 24)

> Initialises the LoRA adapter. This sets up the matrices $A$ and $B$ from the equation above. The matrix $A$ is initialised with random weights from a standard normal distribution; the matrix $B$ is initialised with zeros. The argument *pretrained* is the linear layer from the pre-trained model that should be adapted. The arguments *rank* and *alpha* are the rank $r$ and the hyperparameter $\alpha$ in the equation above.

**forward** (*self*, *x*)

> Sends an input *x* through the adapter, implementing the equation above.

### 🎓 Task 4.09: Inject the adapter into the pre-trained model

The final step is to construct an adapted model by injecting the LoRA adapters into the pre-trained model.

In [44]:
def make_lora_model(rank):
    # Load a fresh pretrained DistilBERT model.
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    # Freeze all parameters first.
    for param in model.parameters():
        param.requires_grad = False

    # Extract the query and value layers.
    original_layers = extract(model)

    # Wrap each query/value layer in a LoRA adapter.
    lora_layers = {}

    for name, layer in original_layers.items():
        lora_layers[name] = LoRA(
            pretrained=layer,
            rank=rank,
            alpha=2 * rank
        )

    # Replace the original layers with the LoRA-wrapped versions.
    replace(model, lora_layers)

    # Keep the classification head trainable.
    for param in model.pre_classifier.parameters():
        param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    # Train only the LoRA parameters and classification head.
    return train(model)

Implement the function to match the following specification:

> **make_lora_model** (*rank*)
>
> Returns a model that is identical to the pre-trained model, except that the query and value linear layers have been wrapped in LoRA adapters, and the LoRA adapters and the head layers of the pre-trained model have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.) The rank of the adapters is specified by the argument *rank*. The *alpha* value of the adapters is set to twice the rank (a common rule of thumb).

Run the next cell to evaluate your model for $r = 6$ and $\alpha = 12$. How many trainable parameters does the adapted model have? What accuracy do you get? How do these value relate to the number of trainable parameters and accuracy of the fully fine-tuned model, in terms of percentages?

In [45]:
lora_model = make_lora_model(6)

evaluate(lora_model)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of trainable parameters: 702722


  0%|          | 0/750 [00:00<?, ?it/s]/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  1%|          | 6/750 [00:01<02:11,  5.64it/s]

{'loss': 0.6891, 'grad_norm': 3.312927007675171, 'learning_rate': 4.966666666666667e-05, 'epoch': 0.02}


  1%|▏         | 11/750 [00:02<01:48,  6.81it/s]

...

{'loss': 0.1308, 'grad_norm': 38.452056884765625, 'learning_rate': 0.0, 'epoch': 3.0}



100%|██████████| 750/750 [02:00<00:00,  6.21it/s]
/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.3680651783943176, 'eval_accuracy': 0.89, 'eval_runtime': 4.3773, 'eval_samples_per_second': 114.226, 'eval_steps_per_second': 14.393, 'epoch': 3.0}
{'train_runtime': 120.7514, 'train_samples_per_second': 49.689, 'train_steps_per_second': 6.211, 'train_loss': 0.3280238605539004, 'epoch': 3.0}


100%|██████████| 63/63 [00:04<00:00, 14.65it/s]


{'eval_loss': 0.3680651783943176,
 'eval_accuracy': 0.89,
 'eval_runtime': 4.3736,
 'eval_samples_per_second': 114.322,
 'eval_steps_per_second': 14.405}

#### REMARK - Task 4.09

With rank $r=6$ and $\alpha=12$, the LoRA model achieved an evaluation accuracy of **89.0%** using **702,722 trainable parameters**.

For comparison, full fine-tuning achieved **90.2%** accuracy with **66,955,010 trainable parameters**, while head tuning achieved **81.0%** accuracy with **592,130 trainable parameters**.

Therefore, LoRA uses only about **1.05%** of the trainable parameters required for full fine-tuning, corresponding to a reduction of approximately **99%**, while losing only **1.2 percentage points** in accuracy. It also substantially outperforms head tuning while requiring only a relatively small number of additional trainable parameters.

This shows the main advantage of LoRA: most of the performance of full fine-tuning can be recovered by learning small low-rank updates to selected layers instead of updating all parameters of the pretrained model.

In summary, **full fine-tuning** updates all model parameters, **head tuning** freezes the Transformer and trains only the classification head, while **LoRA** keeps the original Transformer weights frozen and learns small low-rank updates in selected attention layers together with the classification head.

In our lab assignment, LoRA is applied to the **query and value projection matrices in all 6 DistilBERT Transformer blocks**. These projections are shared across the attention heads in each block, so LoRA affects all heads indirectly, while the key projections, attention output projections, and feed-forward layers remain frozen.

## Alternatives to Transformer-based models

Even with methods for parameter-efficient fine-tuning, applying DistilBERT and other Transformer-based models comes at a significant cost – an investment that does not always pay off. In the final task of this lab, we ask you to explore a more traditional approach to classification and contrast it with the pre-training/fine-tuning approach of neural language models.

### 🎓 Task 4.10: Comparing with a non-neural classifier

Browse the web to find a tutorial on how to apply a classifier from the [scikit-learn](https://scikit-learn.org/stable/) library to the problem of sentiment classification and implement the method here in this notebook. We suggest you use [multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) or [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). (Once you have code for one method, it is easy to switch to the other.) Evaluate your chosen classifier on the IMDB dataset.

Questions to consider:

* Which classifier did you try? What results did you get? How long did it take you to train and run the classifier?
* What is your perspective on the trade-off between accuracy and resource requirements between the two approaches?
* What did you learn? How, exactly, did you learn it? Why does this learning matter?

In [46]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Get the raw review text and labels.
X_train = imdb_dataset["train"]["review"]
y_train = imdb_dataset["train"]["label"]

X_eval = imdb_dataset["eval"]["review"]
y_eval = imdb_dataset["eval"]["label"]


# Convert the reviews into TF-IDF feature vectors.
vectorizer = TfidfVectorizer()

start_time = time.time()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_eval_tfidf = vectorizer.transform(X_eval)


# Train a logistic regression classifier.
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_tfidf, y_train)


# Predict sentiment on the evaluation set.
predictions = classifier.predict(X_eval_tfidf)

runtime = time.time() - start_time
eval_accuracy = accuracy_score(y_eval, predictions)


print(f"Accuracy: {eval_accuracy:.3f}")
print(f"Runtime: {runtime:.3f} seconds")

Accuracy: 0.840
Runtime: 0.174 seconds


#### ANSWER - Task 4.10

I used **TF-IDF features together with a Logistic Regression classifier** for sentiment classification. TF-IDF converts each review into a sparse vector based on the importance of the words it contains, while Logistic Regression learns a linear decision boundary between positive and negative reviews.

The classifier achieved an evaluation accuracy of **84.0%** and took only **0.174 seconds** to train and evaluate. This is a strong result considering how simple and computationally cheap the method is.

For comparison, head tuning achieved **81.0%**, LoRA achieved **89.0%**, and full fine-tuning achieved **90.2%**. This means that the non-neural classifier actually outperformed head tuning, while requiring dramatically less computation. However, LoRA and full fine-tuning still achieved better accuracy.

The main trade-off is therefore between **accuracy and resource requirements**. Logistic Regression is extremely fast, lightweight, and easy to train, while Transformer-based approaches require much more computation but can achieve better performance. LoRA provides a useful middle ground because it reaches close to the accuracy of full fine-tuning while updating only a small fraction of the model parameters.

From this task, I learned that simple classical methods can still be very competitive for text classification, especially when the dataset is relatively small and computational efficiency is important. I learned this by directly comparing the accuracy and runtime of the different approaches on the same sentiment classification task. This matters because the most complex model is not always the most practical choice; the appropriate method depends on how much accuracy is needed and what computational resources are available.

**🥳 Congratulations on finishing this lab! 🥳**